In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/kohya-ss/sd-scripts /content/sd-scripts
%cd /content/sd-scripts
!pip install -r requirements.txt -q
!pip install xformers -q

In [ ]:
# Jangan install apapun dulu, langsung cek
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ CPU")

In [ ]:
# ======== CELL 1 — Install PyTorch CUDA yang benar ========
!pip install torch==2.1.0+cu121 torchvision==0.16.0+cu121 \
    --index-url https://download.pytorch.org/whl/cu121 -q
!pip install xformers==0.0.23 -q
print("✅ Selesai install. RESTART RUNTIME sekarang!")

In [ ]:
# Cell 1 — Uninstall PyTorch CPU, install versi CUDA
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

# Verifikasi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ Masih CPU!")

In [ ]:
# Fix versi torchao dan peft
!pip install -q --upgrade torchao
!pip install -q --upgrade peft
!pip install -q --upgrade diffusers

# Restart runtime setelah install
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import os

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

image_dir = "/content/drive/MyDrive/PCD/DaKamasan/datasets"
caption_dir = "/content/drive/MyDrive/PCD/DaKamasan/datasetC"
os.makedirs(caption_dir, exist_ok=True)

for img_file in os.listdir(image_dir):
    img = Image.open(f"{image_dir}/{img_file}").convert("RGB")

    # Auto caption
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)

    # Tambahkan trigger word
    full_caption = f"{caption}, pita maha style, balinese traditional painting"

    # Simpan caption
    name = os.path.splitext(img_file)[0]
    with open(f"{caption_dir}/{name}.txt", "w") as f:
        f.write(full_caption)

print("Captioning selesai!")

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import os

# Load BLIP model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

image_dir = "/content/drive/MyDrive/PCD/DaKamasan/datasetplus"
skipped = 0
processed = 0

for img_file in sorted(os.listdir(image_dir)):
    if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    name = os.path.splitext(img_file)[0]
    caption_path = os.path.join(image_dir, f"{name}.txt")

    # ← SKIP jika caption sudah ada
    if os.path.exists(caption_path):
        skipped += 1
        continue

    # Generate caption untuk gambar baru
    img = Image.open(os.path.join(image_dir, img_file)).convert("RGB")
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(out[0], skip_special_tokens=True)

    # Tambahkan trigger words khas Kamasan
    full_caption = (
        f"{caption}, ptmh style, kamasan balinese painting, "
        f"traditional balinese art, flat perspective, intricate details, "
        f"natural earth tones"
    )

    with open(caption_path, "w") as f:
        f.write(full_caption)

    processed += 1
    print(f"✅ {img_file}: {full_caption}")

print(f"\nSelesai! {processed} caption baru, {skipped} dilewati.")

In [ ]:
# Jalankan cell ini dulu untuk buat config yang benar
import os

# Cek isi folder dataset kamu dulu
dataset_path = "/content/drive/MyDrive/PCD/DaKamasan/datasetplus"
files = os.listdir(dataset_path)

images   = [f for f in files if f.endswith(('.jpg','.png','.jpeg'))]
captions = [f for f in files if f.endswith('.txt')]

print(f"📸 Jumlah gambar  : {len(images)}")
print(f"📝 Jumlah caption : {len(captions)}")
print(f"📁 Path dataset   : {dataset_path}")

# Cek apakah setiap gambar punya caption
missing = []
for img in images:
    name = os.path.splitext(img)[0]
    if f"{name}.txt" not in captions:
        missing.append(img)

if missing:
    print(f"\n⚠️  {len(missing)} gambar TANPA caption:")
    for m in missing[:5]:
        print(f"   - {m}")
else:
    print("\n✅ Semua gambar sudah punya caption!")

In [ ]:
# Buat lora_config.toml yang benar
config = f"""
[general]
enable_bucket = true
shuffle_caption = true
keep_tokens = 2
caption_dropout_rate = 0.05

[[datasets]]
resolution = 512
batch_size = 1
enable_bucket = true
bucket_no_upscale = true

  [[datasets.subsets]]
  image_dir = "{dataset_path}"
  caption_extension = ".txt"
  num_repeats = 12
"""

config_path = "/content/drive/MyDrive/PCD/DaKamasan/LoRA/lora_config.toml"
with open(config_path, "w") as f:
    f.write(config)

print(f"✅ Config tersimpan di: {config_path}")
print(f"\nIsi config:")
print(config)

In [ ]:
# Cell 1 — Uninstall lama, install versi yang kompatibel
!pip uninstall basicsr gfpgan -y -q

# Install basicsr fork yang sudah fix bug ini
!pip install basicsr==1.4.2 -q
!pip install facexlib>=0.2.5 -q
!pip install realesrgan -q

# Patch manual untuk Python 3.12 + torchvision baru
!pip install gfpgan -q --no-deps

# Fix modul yang hilang
import torchvision.transforms.functional as F
import torchvision.transforms
# Buat alias manual
import sys, types
mod = types.ModuleType('torchvision.transforms.functional_tensor')
mod.rgb_to_grayscale = F.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = mod

print("✅ Patch selesai!")

In [ ]:
# Cell 2 — Test import
from gfpgan import GFPGANer
print("✅ GFPGAN berhasil diimport!")

In [ ]:
%%bash
cat > /content/accelerate_config.yaml << EOF
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
fp16: true
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
EOF

accelerate config default --config_file /content/accelerate_config.yaml

In [ ]:
%%bash
cd /content/sd-scripts

accelerate launch --config_file /content/accelerate_config.yaml \
  train_network.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --dataset_config="/content/drive/MyDrive/PCD/DaKamasan/LoRA/lora_config.toml" \
  --output_dir="/content/drive/MyDrive/PCD/DaKamasan/output" \
  --output_name="pita_maha_v3_dim64" \
  \
  --network_module=networks.lora \
  --network_dim=64 \
  --network_alpha=32 \
  \
  --learning_rate=5e-5 \
  --unet_lr=5e-5 \
  --text_encoder_lr=1e-5 \
  --lr_scheduler="cosine_with_restarts" \
  --lr_warmup_steps=100 \
  --lr_scheduler_num_cycles=3 \
  \
  --optimizer_type="AdamW8bit" \
  --train_batch_size=1 \
  --max_train_epochs=15 \
  --save_every_n_epochs=3 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --save_model_as=safetensors \
  --clip_skip=2 \
  --min_snr_gamma=5 \
  --seed=42

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

# ── Load model ──────────────────────────────────────────────
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)

# Upgrade scheduler → hasil lebih tajam & detail
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++",
    use_karras_sigmas=True        # ← smoother, less noise
)

# Load LoRA
pipe.load_lora_weights(
    "/content/drive/MyDrive/PCD/DaKamasan/output/",
    weight_name="pita_maha_v2.safetensors"
)

pipe = pipe.to("cuda")
pipe.enable_attention_slicing()   # hemat VRAM
print("✅ Model siap!")

In [ ]:
# ── Prompt yang jauh lebih detail ───────────────────────────

prompt = """
ptmh style, kamasan balinese painting,
a balinese dancer """

negative_prompt = """
blurry, low quality, deformed face, ugly face, bad anatomy,
extra limbs, mutated hands, poorly drawn face, disfigured,
modern style, photograph, western art, 3d render,
dark background, black background, plain background,
lowres, watermark, signature, jpeg artifacts,
cross-eyed, asymmetrical face
"""

# ── Generate dengan parameter lebih baik ────────────────────
generator = torch.Generator("cuda").manual_seed(torch.randint(0, 999999, (1,)).item())

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=500,        # ← naik dari 30 ke 50 (lebih detail)
    guidance_scale=7.5,            # ← naik dari 7.5 (lebih ikut prompt)
    width=512,
    height=768,                    # ← portrait lebih cocok untuk figur manusia
    generator=generator,
    clip_skip=2,
).images[0]

image.save("/content/drive/MyDrive/PCD/DaKamasan/output/hasil_v2_improved.png")
image

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
import torch
from PIL import Image

# Load Img2Img pipeline
pipe_i2i = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe_i2i.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe_i2i.scheduler.config,
    use_karras_sigmas=True
)
pipe_i2i.load_lora_weights(
    "/content/drive/MyDrive/PCD/DaKamasan/output/",
    weight_name="pita_maha_v2.safetensors"
)
pipe_i2i = pipe_i2i.to("cuda")

# Load gambar sebelumnya sebagai input
init_image = Image.open("/content/drive/MyDrive/PCD/DaKamasan/output/hasil_v2_improved.png")
init_image = init_image.resize((512, 768))

prompt = """ptmh style, kamasan balinese painting,
beautiful symmetrical face, balinese traditional outfit
almond shaped eyes, defined facial features,
balinese temple garden background, lush tropical trees,
warm earth tones, masterpiece, high detail"""

negative_prompt = """ugly face, deformed face, blurry face,
asymmetrical eyes, bad anatomy, extra limbs,
modern, photograph, western art, low quality, blurry"""

# strength: 0.3-0.5 = perubahan halus, 0.6-0.8 = perubahan besar
result = pipe_i2i(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    strength=0.35,           # ← kunci! tidak terlalu berubah tapi diperbaiki
    guidance_scale=9.0,
    num_inference_steps=50,
    clip_skip=2,
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]

result.save("/content/drive/MyDrive/PCD/DaKamasan/output/hasil_img2img.png")
result

In [ ]:
# Install face restoration tool
!pip install gfpgan -q

from gfpgan import GFPGANer
import cv2
import numpy as np

# Download model GFPGAN
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth \
     -P /content/

# Load restorer
restorer = GFPGANer(
    model_path='/content/GFPGANv1.4.pth',
    upscale=2,                    # ← sekaligus upscale 2x
    arch='clean',
    channel_multiplier=2
)

# Perbaiki wajah dari hasil generate
input_img = cv2.imread("/content/drive/MyDrive/PCD/DaKamasan/output/hasil_v2_improved.png")
_, restored_faces, restored_img = restorer.enhance(
    input_img,
    has_aligned=False,
    only_center_face=False,       # ← perbaiki SEMUA wajah dalam gambar
    paste_back=True
)

# Simpan hasil
output_path = "/content/drive/MyDrive/PCD/DaKamasan/output/hasil_v2_face_fixed.png"
cv2.imwrite(output_path, restored_img)

# Tampilkan perbandingan
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Sebelum (wajah asli)", fontsize=14)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(restored_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Sesudah GFPGAN (wajah diperbaiki)", fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/PCD/DaKamasan/output/comparison.png")
plt.show()

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# Load base model + LoRA kamu
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe.load_lora_weights("/content/drive/MyDrive/PCD/DaKamasan/output/pita_maha_v2.safetensors")
pipe = pipe.to("cuda")

# Generate!
prompt = "a balinese dancer performing legong, pita maha style, intricate details, warm colors, and balinese terrace"
negative_prompt = "blurry, modern, photograph, western art"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    width=768, height=768
).images[0]

image.save("hasil_pita_maha.png")
image

In [ ]:
prompt = "a balinese kamasan painting of balinese woman, pita maha style, intricate details, warm earth tones, flat perspective"
negative_prompt = "blurry, modern, photograph, 3d render, western art, low quality"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    width=768,
    height=768
).images[0]

image.save("/content/drive/MyDrive/PCD/DaKamasan/output/test_generate.png")
image  # tampilkan langsung di Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')